In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-terrain2-kp2000kd50-relinvel20'  # ckpt = 20000
exp_name = 'friction-walking-terrain1-kp4000kd50-linvel20-correct0.1-angvel4-plus-0.5'  # ckpt = 20000
# exp_name = 'friction-walking-part-kp2000kd50'
# exp_name = 'correct-walking-flat-kp2000kd50-resume'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.9
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 4000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.9,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.1, 2.0],
  'restitution': [0.0, 0.2],
  'kp': [15000.0, 25000.0],
  'kd': [40.0, 80.0]

In [10]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [11]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [12]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 2.0723e-01, -1.0675e-01,  2.3348e-01, -2.3653e-01, -9.2841e-01,
          5.8112e-02, -1.5232e-01,  2.5601e-04,  3.3599e-01, -2.5081e-01,
         -1.0854e+00,  2.3260e-01]], device='cuda:0')
Scaled actions :  tensor([[ 2.0723e-01, -1.0675e-01,  2.3348e-01, -2.3653e-01, -9.2841e-01,
          5.8112e-02, -1.5232e-01,  2.5601e-04,  3.3599e-01, -2.5081e-01,
         -1.0854e+00,  2.3260e-01]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[ 1.2169e-05, -1.2488e-02,  1.7362e-05,  2.1962e-10, -1.1254e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.1109e-07,
         -1.5817e-07, -4.7266e-05,  2.2542e-04, -1.2803e-04,  2.3233e-07,
          1.6094e-07, -7.5484e-09, -4.7386e-05,  2.2554e-04, -1.2821e-04,
         -1.3234e-06, -5.5543e-06, -7.9086e-06, -2.3640e-03,  1.1269e-02,
         -6.4021e-03,  1.1617e-05,  8.0468e-06, -3.7742e-07, -2.3695e-03,
          1.1281e-02, -6.4100e-03, -6.6171e-05,  2.0723e-01, -1.0675e-01,
          2.3348e-01, -2.3653e-01, -9.2841e-01,  5.8112e-02, -1.5232e-01,
          2.5601e-04,  3.3599e-01, -2.5081e-01, -1.0854e+00,  2.3260e-01]],
       device='cuda:0')
torques: [ 7.82998753e-17 -1.91648509e-16 -2.29733190e-05  5.30499604e-05
 -2.16353278e-05  2.76383991e-16 -7.71195755e-17 -7.09450799e-17
 -2.29733190e-05  5.30499604e-05 -2.16353278e-05 -4.93332534e-17]
データ収集: step 2


In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.0859,  0.0024,  0.0853, -0.3141, -1.2818, -0.2398, -0.1900,  0.2185,
          0.2453,  0.1050, -1.2366,  0.4718]], device='cuda:0')
Scaled actions :  tensor([[ 0.0859,  0.0024,  0.0853, -0.3141, -1.2818, -0.2398, -0.1900,  0.2185,
          0.2453,  0.1050, -1.2366,  0.4718]], device='cuda:0')
obs :  tensor([[ 8.4505e-02, -2.3680e-01,  3.5133e-02, -5.5201e-03, -2.0533e-03,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  9.5315e-03,
         -3.6720e-03,  1.0663e-02, -1.2982e-04, -1.2180e-02,  1.0616e-02,
         -9.4230e-03, -5.4202e-03,  8.5388e-03,  1.0828e-03, -1.2431e-02,
          8.8277e-03,  8.6853e-02, -3.1736e-02,  9.9618e-02, -1.3416e-02,
         -1.0445e-01,  8.7548e-02, -8.5269e-02, -4.6923e-02,  7.9900e-02,
         -1.9174e-03, -1.0666e-01,  7.9967e-02,  8.5879e-02,  2.4479e-03,
          8.5305e-02, -3.1412e-01, -1.2818e+00, -2.3981e-01, -1.8997e-01,
          2.1850e-01,  2.4531e-01,  1.0502e-01, -1.2366e+00,  4.7

In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.0509, -0.0415, -0.6954, -0.4256, -1.0335, -0.4766,  0.1940,  0.2364,
         -0.2292, -0.2206, -0.9951,  0.4408]], device='cuda:0')
Scaled actions :  tensor([[-0.0509, -0.0415, -0.6954, -0.4256, -1.0335, -0.4766,  0.1940,  0.2364,
         -0.2292, -0.2206, -0.9951,  0.4408]], device='cuda:0')
obs :  tensor([[ 5.4523e-02, -4.4365e-01, -1.7237e-02, -1.9609e-02, -4.8441e-03,
         -9.9980e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.4944e-02,
         -4.8941e-03,  4.0324e-02, -4.0740e-03, -4.3852e-02,  1.5276e-02,
         -3.7616e-02, -9.4871e-03,  3.1641e-02,  7.0388e-03, -4.6079e-02,
          3.6807e-02,  1.5860e-01,  1.3587e-02,  1.8638e-01, -2.5017e-02,
         -2.0258e-01, -2.9732e-02, -1.8653e-01,  6.3307e-04,  1.4450e-01,
          5.5494e-02, -2.1861e-01,  1.8892e-01, -5.0872e-02, -4.1510e-02,
         -6.9543e-01, -4.2560e-01, -1.0335e+00, -4.7655e-01,  1.9402e-01,
          2.3644e-01, -2.2918e-01, -2.2059e-01, -9.9512e-01,  4.4

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.2774, -0.1067, -0.3901, -0.7229, -0.8767, -0.4058,  0.1986,  0.0338,
         -0.1944, -0.2417, -0.8658,  0.2639]], device='cuda:0')
Scaled actions :  tensor([[-0.2774, -0.1067, -0.3901, -0.7229, -0.8767, -0.4058,  0.1986,  0.0338,
         -0.1944, -0.2417, -0.8658,  0.2639]], device='cuda:0')
obs :  tensor([[ 0.0264, -0.1698, -0.0189, -0.0311, -0.0076, -0.9995,  1.0000,  0.0000,
          0.0000,  0.0548, -0.0033,  0.0696, -0.0103, -0.0951, -0.0034, -0.0639,
         -0.0067,  0.0525,  0.0110, -0.1017,  0.0863,  0.0500,  0.0098,  0.1148,
         -0.0393, -0.3003, -0.1455, -0.0773,  0.0094,  0.0718,  0.0297, -0.3198,
          0.2805, -0.2774, -0.1067, -0.3901, -0.7229, -0.8767, -0.4058,  0.1986,
          0.0338, -0.1944, -0.2417, -0.8658,  0.2639]], device='cuda:0')
torques: [-200.         -156.81641839 -200.         -200.         -200.
 -200.          200.          200.         -200.         -200.
 -200.          200.        ]
データ収集: step 5


In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 0.1251,  0.1313,  0.1325, -0.5670, -0.8358,  0.0285, -0.0752,  0.0672,
          0.3959, -0.0151, -0.9876,  0.3018]], device='cuda:0')
Scaled actions :  tensor([[ 0.1251,  0.1313,  0.1325, -0.5670, -0.8358,  0.0285, -0.0752,  0.0672,
          0.3959, -0.0151, -0.9876,  0.3018]], device='cuda:0')
obs :  tensor([[ 5.7385e-02,  9.8419e-02, -3.7234e-02, -3.2535e-02, -9.7494e-03,
         -9.9942e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  5.2645e-02,
         -2.5553e-03,  8.4040e-02, -1.2095e-02, -1.6479e-01, -4.0294e-02,
         -6.7026e-02, -6.1471e-03,  5.9883e-02,  1.5318e-02, -1.7642e-01,
          1.4982e-01, -5.8830e-02, -2.8481e-03,  4.4175e-02, -9.4132e-03,
         -3.9035e-01, -2.2336e-01,  3.4146e-02,  1.2978e-04,  5.1337e-03,
          1.7039e-02, -4.1732e-01,  3.4028e-01,  1.2514e-01,  1.3131e-01,
          1.3254e-01, -5.6704e-01, -8.3581e-01,  2.8483e-02, -7.5184e-02,
          6.7194e-02,  3.9587e-01, -1.5141e-02, -9.8756e-01,  3.0

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.0554, -0.0305,  0.1260, -0.6114, -0.6422,  0.0588, -0.2590, -0.1268,
          0.4300, -0.0816, -0.8685,  0.2644]], device='cuda:0')
Scaled actions :  tensor([[ 0.0554, -0.0305,  0.1260, -0.6114, -0.6422,  0.0588, -0.2590, -0.1268,
          0.4300, -0.0816, -0.8685,  0.2644]], device='cuda:0')
obs :  tensor([[-0.1669, -0.0357, -0.0092, -0.0318, -0.0070, -0.9995,  1.0000,  0.0000,
          0.0000,  0.0525,  0.0053,  0.0999, -0.0172, -0.2545, -0.0719, -0.0626,
          0.0014,  0.0716,  0.0143, -0.2715,  0.2268,  0.0475,  0.0719,  0.1030,
         -0.0374, -0.4953, -0.1048,  0.0099,  0.0700,  0.0990, -0.0262, -0.5238,
          0.3919,  0.0554, -0.0305,  0.1260, -0.6114, -0.6422,  0.0588, -0.2590,
         -0.1268,  0.4300, -0.0816, -0.8685,  0.2644]], device='cuda:0')
torques: [ 200.          200.           52.38227145 -200.         -200.
  200.          -80.58798951  150.56234628  200.          -84.273108
 -200.         -200.        ]
データ収集: s

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[-0.2086, -0.4589, -0.4498, -0.4943, -0.4378,  0.0234, -0.0274, -0.3711,
         -0.1104, -0.1259, -0.6024,  0.0148]], device='cuda:0')
Scaled actions :  tensor([[-0.2086, -0.4589, -0.4498, -0.4943, -0.4378,  0.0234, -0.0274, -0.3711,
         -0.1104, -0.1259, -0.6024,  0.0148]], device='cuda:0')
obs :  tensor([[-0.0149, -0.0648,  0.1867, -0.0343, -0.0036, -0.9994,  1.0000,  0.0000,
          0.0000,  0.0601,  0.0129,  0.1252, -0.0301, -0.3645, -0.0796, -0.0711,
          0.0043,  0.1002,  0.0037, -0.3878,  0.2989,  0.0307,  0.0111,  0.1351,
         -0.0850, -0.5905,  0.0155, -0.0857, -0.0301,  0.1753, -0.0765, -0.6299,
          0.3064, -0.2086, -0.4589, -0.4498, -0.4943, -0.4378,  0.0234, -0.0274,
         -0.3711, -0.1104, -0.1259, -0.6024,  0.0148]], device='cuda:0')
torques: [ -13.55598766 -171.39516568 -121.1770702  -200.         -200.
  200.         -200.         -200.          200.         -200.
 -200.         -200.        ]
データ収集: step 8


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.1460,  0.7072,  0.4445, -0.6490, -0.1728,  0.3270,  0.1450,  0.6033,
          0.5582, -0.2612, -0.6059,  0.3602]], device='cuda:0')
Scaled actions :  tensor([[ 0.1460,  0.7072,  0.4445, -0.6490, -0.1728,  0.3270,  0.1450,  0.6033,
          0.5582, -0.2612, -0.6059,  0.3602]], device='cuda:0')
obs :  tensor([[ 0.1993,  0.2138,  0.1775, -0.0307, -0.0073, -0.9995,  1.0000,  0.0000,
          0.0000,  0.0553,  0.0058,  0.1457, -0.0521, -0.4846, -0.0635, -0.0786,
         -0.0094,  0.1271, -0.0144, -0.5141,  0.3478, -0.0675, -0.0764,  0.0760,
         -0.1328, -0.5610,  0.1323,  0.0018, -0.0937,  0.0997, -0.1007, -0.6009,
          0.1971,  0.1460,  0.7072,  0.4445, -0.6490, -0.1728,  0.3270,  0.1450,
          0.6033,  0.5582, -0.2612, -0.6059,  0.3602]], device='cuda:0')
torques: [-200. -200. -200. -200.  200.  200.  200. -200. -200. -200.  200. -200.]
データ収集: step 9


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.1840, -0.2349,  0.0962, -0.0598,  0.2104,  0.4714, -0.0601, -0.6495,
          0.5960, -0.2428, -0.2274, -0.0640]], device='cuda:0')
Scaled actions :  tensor([[-0.1840, -0.2349,  0.0962, -0.0598,  0.2104,  0.4714, -0.0601, -0.6495,
          0.5960, -0.2428, -0.2274, -0.0640]], device='cuda:0')
obs :  tensor([[-0.0025,  0.0128,  0.0794, -0.0262, -0.0108, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0541, -0.0019,  0.1714, -0.0844, -0.5846, -0.0246, -0.0674,
         -0.0173,  0.1577, -0.0418, -0.6321,  0.3815,  0.0431, -0.0084,  0.1676,
         -0.1833, -0.4530,  0.2449,  0.1017,  0.0026,  0.1950, -0.1683, -0.5546,
          0.1260, -0.1840, -0.2349,  0.0962, -0.0598,  0.2104,  0.4714, -0.0601,
         -0.6495,  0.5960, -0.2428, -0.2274, -0.0640]], device='cuda:0')
torques: [ 200.  200.  200. -200.  200.  200.  200.  200.  200. -200.  200. -200.]
データ収集: step 10


In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.2079,  0.7146, -0.2657, -0.4688,  0.1497,  0.2842,  0.0021,  0.6504,
         -0.2159, -0.0737, -0.2479,  0.2177]], device='cuda:0')
Scaled actions :  tensor([[-0.2079,  0.7146, -0.2657, -0.4688,  0.1497,  0.2842,  0.0021,  0.6504,
         -0.2159, -0.0737, -0.2479,  0.2177]], device='cuda:0')
obs :  tensor([[ 0.1632, -0.0796,  0.2852, -0.0277, -0.0141, -0.9995,  1.0000,  0.0000,
          0.0000,  0.0510, -0.0120,  0.1997, -0.1125, -0.6629,  0.0365, -0.0516,
         -0.0216,  0.2005, -0.0784, -0.7334,  0.3952, -0.0638, -0.0851,  0.1200,
         -0.0999, -0.3403,  0.3556,  0.0551, -0.0412,  0.2297, -0.1935, -0.4687,
          0.0216, -0.2079,  0.7146, -0.2657, -0.4688,  0.1497,  0.2842,  0.0021,
          0.6504, -0.2159, -0.0737, -0.2479,  0.2177]], device='cuda:0')
torques: [-200.         -200.         -200.          200.          200.
  200.         -160.59930866 -200.          200.         -200.
  200.         -200.        ]
データ収集: step 1

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 12
Original actions :  tensor([[ 0.0477,  0.9500, -0.1858, -0.6117,  0.1748,  0.2274, -0.0051,  0.5893,
          0.1575, -0.4419,  0.1603,  0.4154]], device='cuda:0')
Scaled actions :  tensor([[ 0.0477,  0.9500, -0.1858, -0.6117,  0.1748,  0.2274, -0.0051,  0.5893,
          0.1575, -0.4419,  0.1603,  0.4154]], device='cuda:0')
obs :  tensor([[ 0.2088,  0.1651,  0.4332, -0.0203, -0.0198, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0061, -0.0411,  0.2270, -0.1594, -0.7513,  0.2113, -0.0188,
         -0.0199,  0.2666, -0.1416, -0.8742,  0.3571, -0.0676, -0.1252,  0.0471,
         -0.1234, -0.1120,  0.5108,  0.0283, -0.0383,  0.1649, -0.1402, -0.2472,
         -0.2007,  0.0477,  0.9500, -0.1858, -0.6117,  0.1748,  0.2274, -0.0051,
          0.5893,  0.1575, -0.4419,  0.1603,  0.4154]], device='cuda:0')
torques: [ 200.         -200.         -200.         -200.          200.
  200.         -158.85944488 -200.          200.          132.55862648
  200.         -200.        ]
データ収集

In [46]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=2.178, Scaled action max=2.178
Step 1/10, Total steps: 213
steps: 213
actions : tensor([[-0.5843, -0.8207,  1.8814,  0.6421,  2.1778,  0.7111,  0.0231,  0.3438,
          0.8254,  0.1639,  0.4759, -1.0029]], device='cuda:0')
target_dof_pos: tensor([[-0.5974, -0.7513,  1.1840,  2.1879,  1.1615,  0.6612,  0.0504,  0.4807,
          0.0613,  1.6673, -0.2377, -0.9404]], device='cuda:0')
Step 1: Original action max=2.316, Scaled action max=2.316
Step 2: Original action max=2.437, Scaled action max=2.437
データ収集完了: 10 steps collected with action_scale=1.0


In [47]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [48]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
